<a href="https://colab.research.google.com/github/replyeshab/CineAI-AI-Based-Hybrid-Recommendation-System/blob/main/movie_recommendation_system_Popularity_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Popularity Recommender

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATASET_PATH = "/content/drive/MyDrive/ml-32m/ml-32m"

In [6]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [5]:
movie_stats = pd.read_pickle("/content/drive/MyDrive/Artifact/movie_stats.pkl")
movie_data = pd.read_pickle("/content/drive/MyDrive/Artifact/movie_data.pkl")

In [7]:
def calculate_imdb_score(df, quantile=0.90):
    """
    Calculates IMDb Weighted Rating.

    Parameters
    ----------
    df : DataFrame
    quantile : float
        Minimum vote percentile.

    Returns
    -------
    DataFrame
    """

    df = df.copy()

    C = df["average_rating"].mean()

    m = df["rating_count"].quantile(quantile)

    df["weighted_rating"] = (
        (
            df["rating_count"] /
            (df["rating_count"] + m)
        ) * df["average_rating"]
    ) + (
        (
            m /
            (df["rating_count"] + m)
        ) * C
    )

    return df

In [8]:
popular_movies = calculate_imdb_score(movie_stats)

In [9]:
popular_movies = (
    popular_movies
    .sort_values(
        by="weighted_rating",
        ascending=False
    )
    .reset_index(drop=True)
)

In [10]:
def recommend_popular(top_n=10):

    columns = [
        "clean_title",
        "genres",
        "average_rating",
        "rating_count",
        "weighted_rating"
    ]

    return popular_movies[columns].head(top_n)

In [11]:
def recommend_by_genre(
    genre,
    top_n=10
):

    genre_movies = popular_movies[
        popular_movies["genres"]
        .str.contains(
            genre,
            case=False,
            na=False
        )
    ]

    columns = [
        "clean_title",
        "genres",
        "average_rating",
        "rating_count",
        "weighted_rating"
    ]

    return genre_movies[columns].head(top_n)

In [12]:
recommend_popular()

,clean_title,genres,average_rating,rating_count,weighted_rating
0,"Shawshank Redemption, The",Crime|Drama,4.404614,102929,4.401238
1,Planet Earth,Documentary,4.444369,2948,4.332311
2,"Godfather, The",Crime|Drama,4.317030,66440,4.312134
3,Band of Brothers,Action|Drama|War,4.426539,2811,4.310914
4,Parasite,Comedy|Drama,4.312254,11670,4.284956
5,Planet Earth II,Documentary,4.446830,1956,4.284079
6,"Usual Suspects, The",Crime|Mystery|Thriller,4.265070,67750,4.260458
7,"Godfather: Part II, The",Crime|Drama,4.264468,43111,4.257239
8,12 Angry Men,Drama,4.265311,21863,4.251126
9,Schindler's List,Drama|War,4.236990,73849,4.232852


In [13]:
recommend_by_genre("Sci-Fi")

,clean_title,genres,average_rating,rating_count,weighted_rating
24,Spider-Man: Into the Spider-Verse,Action|Adventure|Animation|Sci-Fi,4.185260,10434,4.157763
25,"Matrix, The",Action|Sci-Fi|Thriller,4.156437,93808,4.153390
27,Inception,Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,4.157170,57931,4.152241
40,Star Wars: Episode V - The Empire Strikes Back,Action|Adventure|Sci-Fi,4.130352,72151,4.126483
42,Interstellar,Sci-Fi|IMAX,4.133447,37157,4.125939
55,"Prestige, The",Drama|Mystery|Sci-Fi|Thriller,4.117731,32965,4.109393
58,Blade Runner,Action|Sci-Fi|Thriller,4.110005,46289,4.104095
64,Star Wars: Episode IV - A New Hope,Action|Adventure|Sci-Fi,4.099824,85010,4.096628
84,Nausicaä of the Valley of the Wind (Kaze no ta...,Adventure|Animation|Drama|Fantasy|Sci-Fi,4.108902,7392,4.072945
96,Alien,Horror|Sci-Fi,4.068401,45211,4.062580


In [14]:
recommend_by_genre("Comedy")

,clean_title,genres,average_rating,rating_count,weighted_rating
4,Parasite,Comedy|Drama,4.312254,11670,4.284956
15,Pulp Fiction,Comedy|Crime|Drama|Thriller,4.196969,98409,4.193962
16,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.202680,32830,4.193669
33,Life Is Beautiful (La Vita è bella),Comedy|Drama|Romance|War,4.150374,29819,4.140893
37,Monty Python and the Holy Grail,Adventure|Comedy|Fantasy,4.135633,46508,4.129614
38,"Sting, The",Comedy|Crime,4.143218,18416,4.128041
50,"Princess Bride, The",Action|Adventure|Comedy|Fantasy|Romance,4.119109,46957,4.113235
52,Fargo,Comedy|Crime|Drama|Thriller,4.116533,58031,4.111786
59,Intouchables,Comedy|Drama,4.118810,18416,4.103958
65,Wallace & Gromit: The Wrong Trousers,Animation|Children|Comedy|Crime,4.111800,17652,4.096412


In [15]:
popular_movies.to_pickle(
    "/content/drive/MyDrive/Artifact/popular_movies.pkl"
)

In [16]:
config = {
    "method": "IMDb Weighted Rating",
    "quantile": 0.90
}

with open(
    "/content/drive/MyDrive/Artifact/popularity_config.pkl",
    "wb"
) as f:

    pickle.dump(config, f)